In [5]:
# 1. INSTALL DEPENDENCIES
!pip install tree-sitter==0.21.3 tree-sitter-languages==1.10.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 148.3 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:

import os
import gc
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import tree_sitter_languages
from torch.utils.data import WeightedRandomSampler, DataLoader
from datasets import Dataset, concatenate_datasets
from transformers import (
    RobertaTokenizer, RobertaModel, RobertaPreTrainedModel,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive
import warnings

warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

MODEL_CHECKPOINT = "microsoft/unixcoder-base"
FOLDER_NAME = "UnixCoder_TaskC_Augmented_Final"
SUB_PREFIX = "submission_TaskC"

FIXED_LEARNING_RATE = 3e-5
MAX_LENGTH = 512
BATCH_SIZE = 128

LOGGING_STEPS = 2000
EVAL_STEPS = 2000
SAVE_STEPS = 2000

BASE_DIR = "/content/drive/MyDrive/Task C"
JSON_PATH = "/content/drive/MyDrive/Task C/hard_negative_indices.json"
OUTPUT_ROOT = os.path.join(BASE_DIR, FOLDER_NAME)

os.makedirs(BASE_DIR, exist_ok=True)

# --- 2. ADVANCED CANONICALIZER ---
class UniversalCanonicalizer:
    def __init__(self):
        self.lang_map = {
            'python': 'python', 'py': 'python', 'java': 'java', 'cpp': 'cpp',
            'c++': 'cpp', 'c': 'c', 'c#': 'c_sharp', 'cs': 'c_sharp',
            'javascript': 'javascript', 'js': 'javascript', 'php': 'php', 'go': 'go'
        }
        self.parsers = {}

    def _get_parser(self, lang_key):
        parser_name = self.lang_map.get(lang_key, 'python')
        if parser_name in self.parsers: return self.parsers[parser_name]
        try:
            parser = tree_sitter_languages.get_parser(parser_name)
            self.parsers[parser_name] = parser
            return parser
        except: return None

    def canonicalize(self, code_str, lang_label):
        if not isinstance(code_str, str) or code_str == "": return ""
        lang_key = str(lang_label).lower()
        parser = self._get_parser(lang_key)
        if not parser: return code_str

        code_bytes = code_str.encode('utf8')
        try:
            tree = parser.parse(code_bytes)
            code_bytes = self._rename_identifiers(code_bytes, tree.root_node)
            canon_code = code_bytes.decode('utf8', errors='ignore')
            return self._flatten_layout(canon_code, self.lang_map.get(lang_key, 'python'))
        except: return code_str

    def _rename_identifiers(self, code_bytes, root_node):
        replacements = []
        counters = {'VAR': 0, 'FUNC': 0, 'TYPE': 0}
        name_map = {}
        keywords = {'if', 'else', 'for', 'while', 'return', 'class', 'def', 'void', 'int', 'float', 'double', 'string', 'public', 'private', 'static', 'import', 'from', 'include', 'var', 'let', 'const', 'try', 'catch', 'true', 'false', 'null'}

        def traverse(node):
            is_id = ("identifier" in node.type or "variable" in node.type or "name" in node.type)
            if is_id and len(node.children) == 0:
                original_name = code_bytes[node.start_byte:node.end_byte].decode('utf8', errors='ignore')
                if original_name not in keywords and len(original_name) > 1:
                    parent_type = node.parent.type if node.parent else ""
                    category = 'VAR'
                    if ('function' in parent_type or 'method' in parent_type or 'call' in parent_type): category = 'FUNC'
                    elif ('class' in parent_type or 'type' in parent_type): category = 'TYPE'

                    if original_name not in name_map:
                        name_map[original_name] = f"{category}_{counters[category]}"
                        counters[category] += 1
                    replacements.append((node.start_byte, node.end_byte, name_map[original_name]))
            for child in node.children: traverse(child)

        traverse(root_node)
        replacements.sort(key=lambda x: x[0], reverse=True)
        mutable_code = bytearray(code_bytes)
        for start, end, new_text in replacements: mutable_code[start:end] = new_text.encode('utf8')
        return mutable_code

    def _flatten_layout(self, code_str, lang_type):
        lines = code_str.split('\n')
        processed_lines = []
        comment_pattern = re.compile(r'//.*|#.*|/\*.*?\*/', re.DOTALL)
        for line in lines:
            line = re.sub(comment_pattern, '', line)
            line = line.replace('\t', '    ').rstrip() if 'python' in lang_type else line.strip()
            if line: processed_lines.append(line)
        return "\n".join(processed_lines)

# --- 3. AUGMENTATION PIPELINE (DATA DOUBLING) ---
class AugmentationPipeline:
    def __init__(self, aug_prob_hard=1.0, aug_prob_easy=0.3):
        self.engine = UniversalCanonicalizer()
        self.aug_prob_hard = aug_prob_hard
        self.aug_prob_easy = aug_prob_easy

    def process_batch(self, examples):
        out_codes, out_labels, out_weights = [], [], []
        for i in range(len(examples['code'])):
            orig, label = examples['code'][i], examples['label'][i]
            is_hard = examples['is_hard'][i] if 'is_hard' in examples else False
            lang = examples['language'][i] if 'language' in examples else 'python'

            # Task C Weights: Hybrid (2) and Adversarial (3) get higher base weight
            # 0: Human, 1: AI, 2: Hybrid, 3: Adversarial
            base_w = {0: 1.0, 1: 2, 2: 5.0, 3: 4}.get(label, 1.0)
            weight = base_w * 5.0 if is_hard else base_w

            # 1. Add Original
            out_codes.append(orig)
            out_labels.append(label)
            out_weights.append(weight)

            # 2. Add Augmented (Canonical) version
            threshold = self.aug_prob_hard if is_hard else self.aug_prob_easy
            if np.random.random() < threshold:
                canon_code = self.engine.canonicalize(orig, lang)
                if canon_code and len(canon_code) > 5:
                    out_codes.append(canon_code)
                    out_labels.append(label)
                    out_weights.append(weight)

        return {"code": out_codes, "label": out_labels, "weight": out_weights}

# --- 4. CUSTOM MODEL ---
class CustomUnixCoder(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        seq_out = outputs[0]
        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)
        logits = self.classifier(self.dropout(torch.cat((mean_p, max_p), 1)))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits

class WeightedTrainer(Trainer):
    def __init__(self, *args, custom_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_sampler = custom_sampler

    def get_train_dataloader(self):
        if self.custom_sampler is None: return super().get_train_dataloader()
        return DataLoader(self.train_dataset, batch_size=self.args.train_batch_size,
                          sampler=self.custom_sampler, collate_fn=self.data_collator, num_workers=4)

# --- 5. DATA LOADING & PREPARATION ---
print("[INFO] Loading Tokenizer & Data...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_CHECKPOINT)
aug_pipeline = AugmentationPipeline()

train_df = pd.read_parquet(os.path.join(BASE_DIR, "train.parquet")).dropna(subset=['code', 'label']).reset_index(drop=True)
val_df = pd.read_parquet(os.path.join(BASE_DIR, "validation.parquet")).dropna(subset=['code', 'label']).reset_index(drop=True)
test_df = pd.read_parquet(os.path.join(BASE_DIR, "test.parquet"))

if os.path.exists(JSON_PATH):
    with open(JSON_PATH, "r") as f: hard_indices = set(json.load(f)["hard_indices"])
    train_df['is_hard'] = train_df.index.isin(hard_indices)
else: train_df['is_hard'] = False

if 'language' not in train_df.columns: train_df['language'] = 'python'

print("[INFO] Applying Augmentation (Doubling Data)...")
ds_train_raw = Dataset.from_pandas(train_df).map(
    aug_pipeline.process_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,
    remove_columns=train_df.columns.tolist()
)

def tokenize_fn(x): return tokenizer(x["code"], truncation=True, max_length=MAX_LENGTH)

print("[INFO] Tokenizing Datasets...")
ds_train = ds_train_raw.map(tokenize_fn, batched=True, remove_columns=["code", "weight"])
ds_val = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True, remove_columns=[c for c in val_df.columns if c != 'label'])
ds_test = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True, remove_columns=[c for c in test_df.columns if c != 'label'])

for ds in [ds_train, ds_val, ds_test]:
    if "label" in ds.column_names: ds.rename_column("label", "labels")
    ds.set_format("torch")

# --- 6. TRAINING PHASES ---
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    _, _, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(p.label_ids, preds), 'f1_macro': f1}

def save_submission(model, ds, df, name):
    t = Trainer(model=model, args=TrainingArguments(output_dir="./tmp", per_device_eval_batch_size=BATCH_SIZE, fp16=True, report_to="none"), data_collator=DataCollatorWithPadding(tokenizer))
    preds = np.argmax(t.predict(ds).predictions, axis=1)
    pd.DataFrame({'id': df.get('id', df.get('ID', df.index)), 'label': preds}).to_csv(os.path.join(BASE_DIR, f"{name}.csv"), index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Loading Tokenizer & Data...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Map (num_proc=4):   0%|          | 0/900000 [00:00<?, ? examples/s]

[INFO] Tokenizing Datasets...


Map:   0%|          | 0/1295881 [00:00<?, ? examples/s]

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
class CustomUnixCoder(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        seq_out = outputs[0]
        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)
        logits = self.classifier(self.dropout(torch.cat((mean_p, max_p), 1)))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits

In [ ]:

# Phase 1: Weighted
print("\n🔹 PHASE 1: Weighted Training...")
model = CustomUnixCoder.from_pretrained(MODEL_CHECKPOINT, num_labels=4)
sampler = WeightedRandomSampler(weights=torch.DoubleTensor(ds_train_raw['weight']), num_samples=len(ds_train_raw), replacement=True)

args_p1 = TrainingArguments(output_dir=os.path.join(OUTPUT_ROOT, "p1"), num_train_epochs=3, learning_rate=FIXED_LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, eval_strategy="steps", eval_steps=2500, save_strategy="no", logging_steps=LOGGING_STEPS, fp16=True, report_to="none")
trainer1 = WeightedTrainer(model=model, args=args_p1, train_dataset=ds_train, eval_dataset=ds_val, compute_metrics=compute_metrics, custom_sampler=sampler, data_collator=DataCollatorWithPadding(tokenizer))
trainer1.train()
save_submission(model, ds_test, test_df, f"{SUB_PREFIX}_P1_new_Best")




🔹 PHASE 1: Weighted Training...


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of CustomUnixCoder were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2500,0.882400,0.544969,0.744990,0.680145
5000,0.752200,0.426807,0.811880,0.753896
7500,0.708300,0.419303,0.820905,0.759618
10000,0.630300,0.392936,0.833910,0.777468
12500,0.594100,0.404462,0.838360,0.783922
15000,0.559800,0.361192,0.859620,0.805178
17500,0.528400,0.351687,0.868195,0.810981
20000,0.477700,0.370395,0.863490,0.810307
22500,0.454300,0.363503,0.871440,0.817798
25000,0.431600,0.370483,0.871425,0.819780


In [9]:
model = CustomUnixCoder.from_pretrained("/content/drive/MyDrive/Task C/UnixCoder_TaskC_Augmented_Final/p2/checkpoint-30000", num_labels=4)

# Phase 2: Natural
print("\n🔹 PHASE 2: Natural Training...")
args_p2 = TrainingArguments(output_dir=os.path.join(OUTPUT_ROOT, "p2"), num_train_epochs=2, learning_rate=FIXED_LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, eval_strategy="steps", eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=SAVE_STEPS, logging_steps=LOGGING_STEPS, load_best_model_at_end=True, metric_for_best_model="f1_macro", fp16=True, report_to="none")
trainer2 = Trainer(model=model, args=args_p2, train_dataset=ds_train, eval_dataset=ds_val, compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer))
trainer2.train()
save_submission(model, ds_test, test_df, f"{SUB_PREFIX}_P2_new2_Best")



🔹 PHASE 2: Natural Training...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,0.264700,0.287729,0.907100,0.858479
4000,0.272200,0.281460,0.908510,0.861637
6000,0.267000,0.289912,0.909015,0.861703
8000,0.264900,0.287575,0.912220,0.865964
10000,0.259600,0.303793,0.910030,0.862985
12000,0.205200,0.345374,0.910310,0.863395
14000,0.197200,0.368092,0.910020,0.863253
16000,0.193900,0.400173,0.906520,0.858887
18000,0.188700,0.388836,0.909410,0.862635
20000,0.184700,0.394334,0.909655,0.862689


In [10]:
df=pd.read_csv("/content/drive/MyDrive/Task C/submission_TaskC_P2_new2_Best.csv")

In [11]:
df.rename(columns={"id":"ID"},inplace=True)

In [13]:
df.to_csv("/content/drive/MyDrive/Task C/submission_TaskC_P2_new2_Best_fixed.csv",index=False)

In [9]:
print("A")

A


In [ ]:

# Phase 3: Merge
print("\n🔹 PHASE 3: Full Merge Training...")
combined_ds = concatenate_datasets([ds_train, ds_val]).shuffle(seed=42)
args_p3 = TrainingArguments(output_dir=os.path.join(OUTPUT_ROOT, "p3"), num_train_epochs=1, learning_rate=FIXED_LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, save_strategy="no", logging_steps=LOGGING_STEPS, fp16=True, report_to="none")
trainer3 = Trainer(model=model, args=args_p3, train_dataset=combined_ds, data_collator=DataCollatorWithPadding(tokenizer))
trainer3.train()

save_submission(model, ds_test, test_df, f"{SUB_PREFIX}_P3_Final")
print("✅ Task C Augmented Training Completed.")


🔹 PHASE 3: Full Merge Training...


Step,Training Loss
2000,0.301300
4000,0.297200


In [ ]:
class CustomUnixCoder(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        seq_out = outputs[0]
        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)
        logits = self.classifier(self.dropout(torch.cat((mean_p, max_p), 1)))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits

class WeightedTrainer(Trainer):
    def __init__(self, *args, custom_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_sampler = custom_sampler

    def get_train_dataloader(self):
        if self.custom_sampler is None: return super().get_train_dataloader()
        return DataLoader(self.train_dataset, batch_size=self.args.train_batch_size,
                          sampler=self.custom_sampler, collate_fn=self.data_collator, num_workers=4)

In [ ]:
test_df = pd.read_parquet(os.path.join(BASE_DIR, "test.parquet"))

def tokenize_fn(x): return tokenizer(x["code"], truncation=True, max_length=MAX_LENGTH)


ds_test = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True, remove_columns=[c for c in test_df.columns if c != 'label'])

for ds in [ds_test]:
    if "label" in ds.column_names: ds.rename_column("label", "labels")
    ds.set_format("torch")


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import os
import gc
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import tree_sitter_languages
from torch.utils.data import WeightedRandomSampler, DataLoader
from datasets import Dataset, concatenate_datasets
from transformers import (
    RobertaTokenizer, RobertaModel, RobertaPreTrainedModel,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive
import warnings
BASE_DIR = "/content/drive/MyDrive/Task C"

FIXED_LEARNING_RATE = 3e-5
MAX_LENGTH = 512
BATCH_SIZE = 128

LOGGING_STEPS = 2000
EVAL_STEPS = 2000
SAVE_STEPS = 2000

tokenizer = RobertaTokenizer.from_pretrained("/content/drive/MyDrive/Task C/UnixCoder_TaskC_Augmented_Final/p2/checkpoint-18000")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


model = CustomUnixCoder.from_pretrained("/content/drive/MyDrive/Task C/UnixCoder_TaskC_Augmented_Final/p2/checkpoint-18000", num_labels=4)


temp_trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./tmp", per_device_eval_batch_size=BATCH_SIZE, fp16=True, report_to="none"),
    data_collator=data_collator
)
out = temp_trainer.predict(ds)


probs_path = os.path.join(BASE_DIR, f"P2_18000_probs.npy")
np.save(probs_path, torch.softmax(torch.tensor(out.predictions), dim=-1).numpy())
print(f"   -> Saved probs to {probs_path}")

   -> Saved probs to /content/drive/MyDrive/Task C/P2_18000_probs.npy
